# 10 — Patrones profesionales, optimización y estilo

Las técnicas que distinguen el código de un analista junior del de uno senior: encadenamiento, memoria, visualización de tablas, y los antipatrones más comunes.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df  = pd.read_csv(TRAIN, low_memory=False)
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')


## Method chaining — operaciones en cadena

Encadenar métodos en una sola expresión es más legible y menos propenso a errores que guardar variables intermedias. Python permite saltar línea dentro de paréntesis.

In [ ]:
# Sin encadenamiento — crea variables intermedias innecesarias
tmp1 = df[df['Region'] == 'West']
tmp2 = tmp1.groupby('Category')['Sales'].sum()
tmp3 = tmp2.sort_values(ascending=False)
tmp4 = tmp3.reset_index()

# Con encadenamiento — una sola expresión, sin estado intermedio
resultado = (
    df
    .query("Region == 'West'")
    .groupby('Category')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .assign(revenue_iva=lambda x: (x['Sales'] * 1.21).round(2))
    .rename(columns={'Sales': 'revenue'})
)
print(resultado)


## pipe() — integrar funciones propias en una cadena

In [ ]:
# pipe() pasa el DataFrame como primer argumento de la función
# permite incluir funciones custom dentro de una cadena de métodos

def log_shape(df, etiqueta=''):
    print(f'[{etiqueta}] shape: {df.shape}')
    return df

def eliminar_nulos_criticos(df, cols):
    return df.dropna(subset=cols)

resultado = (
    air
    .pipe(log_shape, 'inicial')
    .pipe(eliminar_nulos_criticos, ['price', 'room_type'])
    .pipe(log_shape, 'tras dropna')
    .query('price < 1000')
    .pipe(log_shape, 'tras filtro precio')
)


## Categoricals — reducir uso de memoria

Las columnas de texto con pocos valores únicos almacenadas como `object` consumen mucha memoria. Convertirlas a `category` puede reducir el uso hasta un 90%.

In [ ]:
# Comparar memoria antes y después
memoria_antes = air.memory_usage(deep=True).sum() / 1024**2

air_opt = air.copy()
cols_categoricas = ['city', 'room_type', 'property_type',
                     'host_response_time', 'neighbourhood']

for col in cols_categoricas:
    if col in air_opt.columns:
        air_opt[col] = air_opt[col].astype('category')

memoria_despues = air_opt.memory_usage(deep=True).sum() / 1024**2

print(f'Memoria antes:  {memoria_antes:.1f} MB')
print(f'Memoria después: {memoria_despues:.1f} MB')
print(f'Reducción: {(1 - memoria_despues/memoria_antes)*100:.0f}%')


## style.format() — tablas con formato legible

`df.style` no modifica los datos — solo cambia cómo se visualizan en Jupyter. Útil para presentar resultados de análisis.

In [ ]:
resumen = (
    df.groupby('Category')['Sales']
    .agg(revenue_total='sum', ticket_medio='mean', num_pedidos='count')
    .reset_index()
)

(
    resumen.style
    .format({
        'revenue_total': '${:,.0f}',
        'ticket_medio':  '${:,.2f}',
        'num_pedidos':   '{:,}',
    })
    .highlight_max(subset=['revenue_total'], color='#d4edda')
    .highlight_min(subset=['ticket_medio'],  color='#f8d7da')
    .set_caption('Ventas por categoría')
)


## Antipatrones más comunes

In [ ]:
# 1 — iterrows() para operaciones que pueden ser vectorizadas
# MAL: extremadamente lento en datasets grandes
# for idx, row in df.iterrows():
#     df.at[idx, 'iva'] = row['Sales'] * 1.21

# BIEN: operación vectorizada
df['iva'] = df['Sales'] * 1.21

# 2 — SettingWithCopyWarning — modificar una copia sin saberlo
# MAL:
# west = df[df['Region'] == 'West']
# west['nueva_col'] = 1   # puede o no modificar df original

# BIEN: .copy() explícito o .loc[] para modificar el original
west = df[df['Region'] == 'West'].copy()
west['nueva_col'] = 1

# 3 — range(len(df)) para iterar
# MAL:
# for i in range(len(df)):
#     print(df.iloc[i]['Sales'])

# BIEN:
# for _, row in df.iterrows():
#     print(row['Sales'])
# O mejor: usar operaciones vectorizadas directamente

print('Antipatrones documentados — ejecutar cada sección para ver el contraste')


## usecols y dtype en read_csv — optimizar la carga

In [ ]:
# Cargar solo columnas necesarias desde el inicio
# Mucho más eficiente que cargar todo y luego filtrar
df_minimo = pd.read_csv(
    TRAIN,
    usecols=['Order Date', 'Region', 'Category', 'Sales'],
    dtype={'Sales': 'float32'},   # float32 ocupa la mitad que float64
    low_memory=False
)

print(f'Memoria carga completa: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'Memoria carga mínima:   {df_minimo.memory_usage(deep=True).sum() / 1024:.1f} KB')


---
## Resumen final del curso

| Nivel | Habilidades |
|-------|-------------|
| **Fundamentos** | Series, DataFrames, iloc/loc, tipos, índices |
| **Carga** | read_csv con parámetros, info/describe, diagnóstico de nulos |
| **Selección** | Máscaras booleanas, isin, between, query, loc |
| **Limpieza** | astype, to_datetime, fillna, dropna, replace, duplicados |
| **Transformaciones** | assign, apply, map, str accessor, dt accessor, cut/qcut |
| **Agregaciones** | groupby, agg, transform, pivot_table, crosstab |
| **Combinación** | concat, merge (inner/left/outer), indicator |
| **Reshaping** | melt, pivot, pivot_table, explode |
| **Series temporales** | dt, to_period, resample, rolling, pct_change, shift |
| **Profesional** | Method chaining, pipe, categoricals, style.format, antipatrones |
